# Azure ML로 PDF 기반 QA 데이터 생성 (Azure-first)

이 노트북은 **Azure Machine Learning Command Job**으로 PDF에서 한국어 Q&A 데이터를 배치 생성합니다.
LLM은 **Azure AI Foundry**(Azure OpenAI 또는 Foundry Agent Service)를 사용하며,
컨테이너는 GitHub Container Registry에 공개된 멀티클라우드 이미지를 그대로 사용합니다.

> 동일한 이미지가 AWS SageMaker Processing에서도 실행됩니다. (멀티클라우드)


## 사전 준비

1. Azure 구독 + **Azure AI Foundry** 프로젝트(모델 배포: 예 `gpt-4o`)
2. Azure ML 워크스페이스 + GPU(또는 CPU) 컴퓨트 클러스터 (예: `gpu-cluster`)
3. `az login` 완료 (로컬), 컴퓨트에는 **Managed Identity** 권장
4. 역할 부여: 컴퓨트/사용자 ID에 **Cognitive Services OpenAI User** (Foundry 리소스 대상)


In [ ]:
# Azure ML / Identity SDK 설치
%pip install -U azure-ai-ml azure-identity

In [ ]:
from azure.ai.ml import MLClient, command, Input, Output
from azure.ai.ml.entities import Environment, Data
from azure.ai.ml.constants import AssetTypes
from azure.identity import DefaultAzureCredential

# 방법 A) config.json 사용:  ml_client = MLClient.from_config(DefaultAzureCredential())
# 방법 B) 명시적 지정:
SUBSCRIPTION_ID = "<your-subscription-id>"
RESOURCE_GROUP  = "<your-resource-group>"
WORKSPACE       = "<your-aml-workspace>"

ml_client = MLClient(
    DefaultAzureCredential(),
    subscription_id=SUBSCRIPTION_ID,
    resource_group_name=RESOURCE_GROUP,
    workspace_name=WORKSPACE,
)
print("Workspace:", ml_client.workspace_name)

## 1. 입력 PDF를 Blob 데이터스토어에 업로드

로컬 `pdf_qa_extraction/data/` 폴더를 URI 폴더 데이터 자산으로 등록합니다.
(기본 `workspaceblobstore`에 업로드됩니다.)

In [ ]:
pdf_data = Data(
    path="../pdf_qa_extraction/data",   # fsi_data.pdf 포함 폴더
    type=AssetTypes.URI_FOLDER,
    name="pdf-qa-input",
    description="PDF2LLM input PDFs",
)
registered = ml_client.data.create_or_update(pdf_data)
pdf_input_uri = registered.id
print("등록된 입력 데이터:", pdf_input_uri)

## 2. Command Job 정의 (공개 GHCR 이미지 사용)

`AZURE_OPENAI_*` 값을 본인 Foundry 리소스에 맞게 수정하세요.
키를 넣지 않으면 컴퓨트의 Managed Identity(Entra ID)로 인증합니다.

In [ ]:
job = command(
    display_name="pdf-qa-extraction",
    experiment_name="pdf2llm-qa",
    environment=Environment(image="ghcr.io/hyeonsangjeon/pdf2llm-tuning-studio/pdf-qa-extractor:latest"),
    compute="gpu-cluster",                 # CPU 클러스터면 table_model=yolox 권장
    code="../pdf_qa_extraction",           # pdf_qa 패키지 + 엔트리포인트 스냅샷 업로드
    command=(
        "python azureml_job.py --provider azure "
        "--input-dir ${{inputs.pdf}} --output-dir ${{outputs.qa}} "
        "--domain 'International Finance' --num_questions 5 "
        "--num_img_questions 1 --table_model yolox"
    ),
    inputs={"pdf": Input(type="uri_folder", path=pdf_input_uri, mode="ro_mount")},
    outputs={"qa": Output(type="uri_folder", mode="rw_mount")},
    environment_variables={
        "LLM_PROVIDER": "azure",
        "AZURE_MODE": "openai",   # 또는 "agent"
        "AZURE_OPENAI_ENDPOINT": "https://<your-foundry-resource>.openai.azure.com/",
        "AZURE_OPENAI_DEPLOYMENT": "gpt-4o",
        "AZURE_OPENAI_API_VERSION": "2024-10-21",
        # Agent 모드로 바꾸려면 위 두 AZURE_OPENAI_* 대신:
        # "AZURE_AI_PROJECT_ENDPOINT": "https://<proj>.services.ai.azure.com/api/projects/<name>",
        # "AZURE_AI_AGENT_MODEL": "gpt-4o",
    },
)
returned = ml_client.jobs.create_or_update(job)
print("제출된 잡:", returned.name)

## 3. 잡 로그 스트리밍

In [ ]:
ml_client.jobs.stream(returned.name)

## 4. 결과 다운로드 (`qa_pairs.jsonl`)

In [ ]:
ml_client.jobs.download(name=returned.name, output_name="qa", download_path="./outputs")
print("다운로드 완료: ./outputs 아래 qa_pairs.jsonl 확인")

In [ ]:
import json, glob
path = sorted(glob.glob("./outputs/**/qa_pairs.jsonl", recursive=True))
if path:
    with open(path[0], encoding="utf-8") as f:
        rows = [json.loads(l) for l in f]
    print(f"총 {len(rows)}개 QA")
    for r in rows[:3]:
        print(r)

## 5. 공급자 전환 (멀티클라우드)

| 목표 | 설정 |
|---|---|
| Azure OpenAI (기본) | `LLM_PROVIDER=azure`, `AZURE_MODE=openai` |
| Foundry Agent | `LLM_PROVIDER=azure`, `AZURE_MODE=agent`, `AZURE_AI_PROJECT_ENDPOINT=...` |
| OpenAI 직접 | `LLM_PROVIDER=openai`, `OPENAI_API_KEY=...` |
| AWS Bedrock | `LLM_PROVIDER=bedrock` (SageMaker/자격증명 필요) |

`environment_variables`만 바꾸면 동일 이미지·동일 코드로 백엔드가 교체됩니다.
